# Session 1 — Python and NumPy for machine learning

One dataset connects the whole session: Python lists → NumPy arrays → linear regression → verification.

By the end, you will have implemented a complete **fit–predict–evaluate** workflow.

## 0. Imports and reproducible data

We own one explicit random-number generator. Re-running this cell recreates exactly the same observations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
x = rng.uniform(0.0, 10.0, size=50)
y = 1.0 + 2.5 * x + rng.normal(0.0, 2.0, size=x.shape)

assert x.shape == y.shape == (50,)
assert np.isfinite(x).all() and np.isfinite(y).all()
print(f"x shape: {x.shape}; y shape: {y.shape}")

In [ ]:
x_list = x.tolist()
y_list = y.tolist()

print(x_list[:3])
print(y_list[:3])

## 1. Pure Python baseline

The MSE function is the student micro-exercise. The remaining functions are a prepared instructor baseline for the regression walkthrough; students are not expected to author all six in 15 minutes.

The formulas use population covariance and variance. Their common factor cancels in the slope.

In [ ]:
def mean_py(values: list[float]) -> float:
    if not values:
        raise ValueError("mean requires at least one value")
    return sum(values) / len(values)


def covariance_py(x_values: list[float], y_values: list[float]) -> float:
    if not x_values or len(x_values) != len(y_values):
        raise ValueError("x and y must have the same non-zero length")
    x_mean = mean_py(x_values)
    y_mean = mean_py(y_values)
    return sum(
        (x_i - x_mean) * (y_i - y_mean)
        for x_i, y_i in zip(x_values, y_values)
    ) / len(x_values)


def variance_py(values: list[float]) -> float:
    values_mean = mean_py(values)
    return sum((value - values_mean) ** 2 for value in values) / len(values)

In [ ]:
def fit_linear_py(
    x_values: list[float], y_values: list[float]
) -> tuple[float, float]:
    x_variance = variance_py(x_values)
    if x_variance == 0.0:
        raise ValueError("cannot fit a slope when every x value is identical")
    slope = covariance_py(x_values, y_values) / x_variance
    intercept = mean_py(y_values) - slope * mean_py(x_values)
    return slope, intercept


def predict_py(
    x_values: list[float], slope: float, intercept: float
) -> list[float]:
    return [intercept + slope * value for value in x_values]


def mse_py(y_true: list[float], y_pred: list[float]) -> float:
    if not y_true or len(y_true) != len(y_pred):
        raise ValueError("targets and predictions must have the same non-zero length")
    return sum(
        (target - prediction) ** 2
        for target, prediction in zip(y_true, y_pred)
    ) / len(y_true)

In [ ]:
slope_py, intercept_py = fit_linear_py(x_list, y_list)
predictions_py = predict_py(x_list, slope_py, intercept_py)
mse_value_py = mse_py(y_list, predictions_py)

assert len(predictions_py) == len(y_list)
assert np.isfinite([slope_py, intercept_py, mse_value_py]).all()
print(f"slope={slope_py:.4f}, intercept={intercept_py:.4f}, mse={mse_value_py:.4f}")

## 2. Shape reasoning and broadcasting

Before looking at the output, predict the shapes of a row, a column, and a one-column matrix.

In [ ]:
points = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 2.0]])

print("points:", points.shape)
print("one row:", points[0].shape)
print("one column:", points[:, 0].shape)
print("one-column matrix:", points[:, :1].shape)

In [ ]:
row_means = points.mean(axis=1, keepdims=True)
centered = points - row_means

assert row_means.shape == (3, 1)
assert centered.shape == points.shape
assert np.allclose(centered.mean(axis=1), 0.0)
centered

### Optional fast-finisher — pairwise squared distances

Attempt this only after the core row-centering checks pass. Do not use scheduled regression time.

For `points.shape == (n, d)`, construct an array with shape `(n, n)` without Python loops.

Checks: the result is symmetric, non-negative, and has a zero diagonal.

In [ ]:
differences = points[:, None, :] - points[None, :, :]
squared_distances = np.sum(differences**2, axis=2)

assert squared_distances.shape == (len(points), len(points))
assert np.allclose(squared_distances, squared_distances.T)
assert np.allclose(np.diag(squared_distances), 0.0)
assert np.all(squared_distances >= 0.0)
squared_distances

## 3. NumPy regression

The implementation keeps the same mathematical contract as the pure-Python version.

In [ ]:
def fit_linear_numpy(
    x_values: np.ndarray, y_values: np.ndarray
) -> tuple[float, float]:
    x_array = np.asarray(x_values, dtype=np.float64)
    y_array = np.asarray(y_values, dtype=np.float64)
    if (
        x_array.ndim != 1
        or y_array.ndim != 1
        or x_array.shape != y_array.shape
        or x_array.size == 0
    ):
        raise ValueError("x and y must be non-empty one-dimensional arrays with equal shape")
    x_variance = np.mean((x_array - x_array.mean()) ** 2)
    if x_variance == 0.0:
        raise ValueError("cannot fit a slope when every x value is identical")
    covariance = np.mean(
        (x_array - x_array.mean()) * (y_array - y_array.mean())
    )
    slope = covariance / x_variance
    intercept = y_array.mean() - slope * x_array.mean()
    return float(slope), float(intercept)


def predict_numpy(
    x_values: np.ndarray, slope: float, intercept: float
) -> np.ndarray:
    return intercept + slope * np.asarray(x_values, dtype=np.float64)


def mse_numpy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_array = np.asarray(y_true, dtype=np.float64)
    predictions = np.asarray(y_pred, dtype=np.float64)
    if y_array.shape != predictions.shape or y_array.size == 0:
        raise ValueError("targets and predictions must have equal non-zero shape")
    return float(np.mean((y_array - predictions) ** 2))

In [ ]:
slope_np, intercept_np = fit_linear_numpy(x, y)
predictions_np = predict_numpy(x, slope_np, intercept_np)
mse_value_np = mse_numpy(y, predictions_np)
baseline_mse = mse_numpy(y, np.full_like(y, y.mean()))

assert predictions_np.shape == y.shape
assert np.isfinite([slope_np, intercept_np, mse_value_np]).all()
assert mse_value_np < baseline_mse
assert np.allclose(
    [slope_np, intercept_np],
    [slope_py, intercept_py],
    rtol=1e-12,
    atol=1e-12,
)
print(f"slope={slope_np:.4f}, intercept={intercept_np:.4f}")
print(f"model MSE={mse_value_np:.4f}; baseline MSE={baseline_mse:.4f}")

## 4. Verify with `numpy.linalg.lstsq`

`lstsq` avoids explicitly forming or inverting the normal-equation matrix.

In [ ]:
design = np.column_stack([x, np.ones_like(x)])
(slope_lstsq, intercept_lstsq), *_ = np.linalg.lstsq(design, y, rcond=None)

assert np.allclose(
    [slope_np, intercept_np],
    [slope_lstsq, intercept_lstsq],
    rtol=1e-12,
    atol=1e-12,
)
print(f"lstsq slope={slope_lstsq:.4f}, intercept={intercept_lstsq:.4f}")

## 5. Visual verification

A plot complements—but does not replace—the numerical checks.

In [ ]:
order = np.argsort(x)
plt.figure(figsize=(8, 4.5))
plt.scatter(x, y, alpha=0.75, label="observations")
plt.plot(x[order], predictions_np[order], color="crimson", linewidth=2.5, label="fit")
plt.axhline(y.mean(), color="gray", linestyle="--", label="mean baseline")
plt.xlabel("x")
plt.ylabel("y")
plt.title("One reproducible fit–predict–evaluate workflow")
plt.legend()
plt.tight_layout()
plt.show()

## Exit check

You should now be able to explain:

1. why `keepdims=True` helps broadcasting;
2. why the RNG is an explicit object;
3. why predictions must have the same shape as targets;
4. why MSE should be compared with a baseline;
5. why `lstsq` is preferable to an explicit matrix inverse.